## Column Transformer 

#### Every feature is transformed differently so we have to handle each column separetly which in turn give different arrays which later we have to stack into a bigger array

#### lets say we have age,city,gender and review feature in our dataset .
1. Age -> Simple Imputer -> array 
2. city and gender -> onehot encoder ->array
3. review -> Ordinal encoder -> array 
4. we get three arrays -> combine into new array This is how we do separetly .
##### Sklearn new class column transformer does all these handling in a single step 


In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv(r"C:\Users\karth\Downloads\covid_toy.csv")

In [4]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
5,84,Female,NaN,Mild,Bangalore,Yes
78,11,Male,100.0,Mild,Bangalore,Yes
82,24,Male,98.0,Mild,Kolkata,Yes
43,22,Female,99.0,Mild,Bangalore,Yes
19,42,Female,NaN,Strong,Bangalore,Yes


In [8]:
## fever simple imputer 
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [6]:
df.city.value_counts()

Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: city, dtype: int64

In [7]:
df.cough.value_counts()

Mild      62
Strong    38
Name: cough, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

#### Transforming each column explicitly 

In [10]:
df.columns 

Index(['age', 'gender', 'fever', 'cough', 'city', 'has_covid'], dtype='object')

In [14]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [11]:
df.gender.value_counts()

Female    59
Male      41
Name: gender, dtype: int64

In [19]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [21]:
df.fever.isnull().sum()

10

In [22]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [23]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [24]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [ ]:
## Concateinating all arrays into one 
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

### Transforming Using Column Transformer 

In [28]:
from sklearn.compose import ColumnTransformer 

In [32]:
## Two parameter transformers column that we want to transform
## Remainder : columns that you dont want to transform either you can "drop" or "passthrough"
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [34]:
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.transform(X_test).shape

(20, 7)